**Lab type:** debug

**Course:** ML201 — Applied Machine Learning

**Lesson:** Feature Selection: Filter, Wrapper, and Embedded Methods

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 1000

# 5 truly predictive features
X_pred, y = make_classification(
    n_samples=n,
    n_features=5,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

# 5 pure noise features
X_noise = np.random.randn(n, 5)

# 3 high-cardinality near-ID columns (integers 0..n)
# These carry no real signal — they are essentially unique row identifiers
X_id = np.column_stack([
    np.random.choice(np.arange(n), size=n, replace=False).reshape(-1, 1),
    np.random.choice(np.arange(n), size=n, replace=False).reshape(-1, 1),
    np.random.choice(np.arange(n), size=n, replace=False).reshape(-1, 1)
])

# 2 redundant features (linear combos of predictive)
X_redundant = np.column_stack([
    X_pred[:, 0] * 0.8 + X_pred[:, 1] * 0.2 + np.random.randn(n) * 0.05,
    X_pred[:, 2] * 0.6 + X_pred[:, 3] * 0.4 + np.random.randn(n) * 0.05
])

X = np.hstack([X_pred, X_noise, X_id, X_redundant])

feature_names = (
    [f"pred_{i}" for i in range(5)] +
    [f"noise_{i}" for i in range(5)] +
    ["customer_id", "session_id", "transaction_id"] +
    ["redundant_0", "redundant_1"]
)

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

print("Feature list:", feature_names)
print(f"\nDataset shape: {X.shape}")
print(f"Class balance — 0: {(y == 0).sum()}, 1: {(y == 1).sum()} ({y.mean()*100:.1f}% positive)")

## Step 1: Feature selection leakage

An analyst wants to select the top 10 features before modelling. They use `SelectKBest` with mutual information to rank features across the full dataset, then split into train and test.

In [ ]:
# Note: uses X, y from Setup

selector = SelectKBest(mutual_info_classif, k=10)
X_selected = selector.fit_transform(X, y)  # <- Bug 1: selector fit before train/test split

X_tr_sel, X_te_sel, y_tr_sel, y_te_sel = train_test_split(
    X_selected, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = LogisticRegression(max_iter=1000, random_state=42)
leaky_model.fit(X_tr_sel, y_tr_sel)
leaky_auc = roc_auc_score(y_te_sel, leaky_model.predict_proba(X_te_sel)[:, 1])
print(f"(Leaky) AUC: {leaky_auc:.4f}")

**Bug 1 Investigation:** The selector calls `fit_transform(X, y)` on the full dataset — including the rows that will become your test set. What information has the selector seen that it should not have? Does the AUC reported after splitting reflect how the model would perform on truly unseen data, or is it optimistically biased? Explain your reasoning.

*(Write your answer here.)*

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What was wrong:** `SelectKBest.fit_transform(X, y)` computed mutual information scores using all 1000 rows — including the 250 that will later become the test set. The selector "saw" the test labels when deciding which 10 features to keep. Feature selection is a form of preprocessing that must be performed inside the train/test boundary.

**Why the AUC is optimistically biased:** The selected features are tailored to signal patterns present in the test set, so the model's performance on that test set is inflated. On a truly independent dataset, the same 10 features would not necessarily be the best ones — the reported AUC does not reflect how the selection pipeline would generalise.

**Correct approach:** Wrap `SelectKBest` inside a `Pipeline` with the model and fit the pipeline only on `X_train`. If using cross-validation, pass the pipeline to `cross_val_score` so the selector is re-fit on each fold's training data independently.

</details>

## Step 2: MDI overranks high-cardinality features

A colleague uses Random Forest's built-in feature importances (mean decrease in impurity, MDI) to rank features and decide which ones to keep. They drop anything outside the top 10.

In [ ]:
# Note: uses X_train_df, X_test_df, y_train, y_test from Setup

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_df, y_train)

importance = pd.Series(
    rf.feature_importances_,
    index=X_train_df.columns
)
top_features = importance.sort_values(ascending=False).head(10).index.tolist()  # <- Bug 2: MDI ranks customer_id highest

print("Top 10 features by MDI importance:")
print(importance.sort_values(ascending=False).head(10).to_string())
print()
print("Selected features include:", top_features)

**Bug 2 Investigation:** Look at where `customer_id`, `session_id`, and `transaction_id` appear in the MDI ranking. These columns contain integers from 0 to ~1000 — essentially unique row identifiers. Why does MDI rank them highly despite them carrying no real signal about the target? Hint: think about how decision trees split on high-cardinality continuous-valued features.

*(Write your answer here.)*

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What was wrong:** MDI (mean decrease in impurity) sums the impurity reduction from every split on a feature across all trees. A feature like `customer_id` with ~1000 unique integer values gives the tree an enormous number of possible split thresholds. The tree can always find a threshold that partitions the data into two nearly-pure groups of a few rows each, producing large impurity reduction — not because the feature predicts the target, but because its high cardinality lets the tree memorise noise. MDI sums all these spurious reductions, inflating the importance score.

**Why the fix works:** Permutation importance shuffles a feature's values on the test set and measures how much the model's AUC drops. `customer_id` in the test set has different integer values than training (sequential integers continue from the training range), so shuffling it changes nothing the model can exploit — its permutation importance is near zero, correctly reflecting that it carries no real signal.

</details>

## Step 3: Scaler fit on full dataset before RFECV

An analyst wants to use RFECV to find the optimal number of features. Before running RFECV, they scale all features. However, they fit the scaler on the full dataset (before any split), so the scaler has seen the test rows.

In [ ]:
# Note: uses X, y from Setup

scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X)  # <- Bug 3: scaler fit on full dataset before split

X_tr_sc, X_te_sc, y_tr_sc, y_te_sc = train_test_split(
    X_all_scaled, y, test_size=0.25, random_state=42, stratify=y
)

rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=StratifiedKFold(5),
    scoring="roc_auc",
    n_jobs=-1
)
rfecv.fit(X_tr_sc, y_tr_sc)

print(f"Optimal number of features: {rfecv.n_features_}")
selected_mask = rfecv.support_
selected = [feature_names[i] for i, s in enumerate(selected_mask) if s]
print(f"Selected features: {selected}")

**Bug 3 Investigation:** Trace the statistics the scaler learns during `fit_transform(X)`. Specifically: the scaler computes a mean and standard deviation for each feature using *all* rows, including those that will later become `X_te_sc`. When RFECV performs its internal cross-validation folds on `X_tr_sc`, are the mean/std values it is working with truly computed from only the training fold? What should those statistics be based on instead?

*(Write your answer here.)*

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What was wrong:** `StandardScaler.fit_transform(X)` computed per-feature means and standard deviations from all 1000 rows — including the 250 that will become the test set, and including every row that will appear in any RFECV validation fold. When RFECV creates internal cross-validation splits on `X_tr_sc`, each validation fold has already been used to compute the scaling statistics. The "unseen" fold data is not truly unseen to the scaler, creating a subtle form of data leakage.

**What the statistics should be based on:** The mean and std for each feature should be derived only from the current fold's training rows. Each RFECV fold should see a scaler fit on that fold's training data alone.

**Correct approach:** Place `StandardScaler` inside a `Pipeline` before the `LogisticRegression` estimator, then pass the pipeline to `RFECV`. sklearn will re-fit the scaler on each fold's training data before transforming the validation fold — ensuring the statistics never incorporate validation-fold information.

</details>

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# Note: uses X, y, X_train, X_test, y_train, y_test, X_train_df, X_test_df, feature_names from Setup

# ---- Fix 1: SelectKBest inside a Pipeline, cross-validated on training data only ----
pipe_select = Pipeline([
    ("selector", SelectKBest(mutual_info_classif, k=10)),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
cv_scores_select = cross_val_score(
    pipe_select, X_train, y_train,
    cv=StratifiedKFold(5), scoring="roc_auc", n_jobs=-1
)
print("Fix 1 — SelectKBest inside Pipeline (5-fold CV on train set):")
print(f"  AUC: {cv_scores_select.mean():.4f} (+/- {cv_scores_select.std():.4f})")
print()

# ---- Fix 2: Permutation importance (test-set-based, unbiased by cardinality) ----
rf_fixed = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fixed.fit(X_train_df, y_train)

perm = permutation_importance(
    rf_fixed, X_test_df, y_test,
    n_repeats=30, random_state=42, scoring="roc_auc", n_jobs=-1
)
perm_series = pd.Series(
    perm.importances_mean,
    index=feature_names
).sort_values(ascending=False)

print("Fix 2 — Permutation importance (top 10, test set):")
print(perm_series.head(10).to_string())
print()
print("customer_id / session_id / transaction_id permutation importance:")
print(perm_series[["customer_id", "session_id", "transaction_id"]].to_string())
print()

# ---- Fix 3: StandardScaler + model in Pipeline, scaler fit per CV fold ----
pipe_rfecv = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
cv_scores_rfecv = cross_val_score(
    pipe_rfecv, X_train, y_train,
    cv=StratifiedKFold(5), scoring="roc_auc", n_jobs=-1
)
print("Fix 3 — StandardScaler inside Pipeline (5-fold CV on train set):")
print(f"  AUC: {cv_scores_rfecv.mean():.4f} (+/- {cv_scores_rfecv.std():.4f})")
print()
print("All three leakage sources have been eliminated.")